# Support Ticket Classification & Prioritization

This notebook demonstrates the end-to-end ML pipeline for classifying support tickets into categories and predicting their priority using Logistic Regression and TF-IDF.

## System Explanations

### How Tickets are Categorized
1. **Text Cleaning:** Raw ticket text is cleaned by removing punctuation, making everything lowercase, and filtering out common English stopwords (e.g., \"the\", \"is\", \"at\").
2. **Feature Extraction:** The cleaned text is transformed into numerical vectors using **TF-IDF** (Term Frequency-Inverse Document Frequency).
3. **Classification Model:** A **Logistic Regression** model is trained to learn which words strongly correlate with categories like 'Billing', 'Technical Issue', 'Account', and 'General Query'.

### How Priority is Decided
Similar to categorization, priority (High, Medium, Low) is predicted using a dedicated **Logistic Regression** model that identifies language patterns associated with urgency (e.g., \"crashing\", \"locked\"). This allows the system to route urgent issues faster.

### Evaluation Results and Insights
Models are evaluated on a 20% test set using standard ML metrics (Accuracy, Precision, Recall, F1-Score, and Confusion Matrix). Using TF-IDF with Logistic Regression establishes a highly effective baseline for operational support pipelines.

In [ ]:
!pip install pandas numpy scikit-learn nltk matplotlib seaborn

## 1. Setup and Data Generation
First, we import the necessary libraries and generate a synthetic dataset.

In [ ]:
import pandas as pd
import numpy as np
import random
import os
import re
import string
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('stopwords', quiet=True)

def generate_synthetic_data(num_records=1000):
    categories = ['Billing', 'Technical Issue', 'Account', 'General Query']
    priorities = ['High', 'Medium', 'Low']
    
    billing_templates = ["I was charged twice", "Invoice incorrect", "Refund please", "Update payment"]
    tech_templates = ["App crashing", "500 error", "Not loading", "API broken"]
    account_templates = ["Reset password", "Change email", "Account locked", "Delete account"]
    general_templates = ["Business hours?", "Feature request", "Documentation", "Pricing?"]
    
    data = []
    for _ in range(num_records):
        category = random.choice(categories)
        if category == 'Technical Issue':
            priority = np.random.choice(priorities, p=[0.6, 0.3, 0.1])
            text = random.choice(tech_templates)
        elif category == 'Billing':
            priority = np.random.choice(priorities, p=[0.5, 0.4, 0.1])
            text = random.choice(billing_templates)
        elif category == 'Account':
            priority = np.random.choice(priorities, p=[0.3, 0.5, 0.2])
            text = random.choice(account_templates)
        else:
            priority = np.random.choice(priorities, p=[0.05, 0.35, 0.6])
            text = random.choice(general_templates)
        data.append({'ticket_text': text, 'category': category, 'priority': priority})
        
    return pd.DataFrame(data)

df = generate_synthetic_data(1000)
df.head()

: 

## 2. Text Preprocessing
Cleaning text by lowercasing, removing punctuation, and filtering stopwords.

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    stop_words = set(stopwords.words('english'))
    words = [word for word in text.split() if word not in stop_words]
    return ' '.join(words).strip()

df['cleaned_text'] = df['ticket_text'].apply(preprocess_text)
df[['ticket_text', 'cleaned_text']].head()

## 3. Feature Extraction & Model Training
We use TF-IDF and Logistic Regression to train Category and Priority classifiers.

In [ ]:
X = df['cleaned_text']
y_cat = df['category']
y_prio = df['priority']

X_train, X_test, y_cat_train, y_cat_test, y_prio_train, y_prio_test = train_test_split(
    X, y_cat, y_prio, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training Category Classifier...")
cat_model = LogisticRegression(max_iter=1000)
cat_model.fit(X_train_tfidf, y_cat_train)

print("Training Priority Classifier...")
prio_model = LogisticRegression(max_iter=1000)
prio_model.fit(X_train_tfidf, y_prio_train)

## 4. Evaluation
Check accuracy, classification report, and confusion matrices.

In [ ]:
def evaluate_model(y_true, y_pred, labels, title):
    print(f"--- {title} ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}\n")
    print(classification_report(y_true, y_pred, labels=labels))
    
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f'Confusion Matrix: {title}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

cat_pred = cat_model.predict(X_test_tfidf)
evaluate_model(y_cat_test, cat_pred, cat_model.classes_, "Category Classifier")

prio_pred = prio_model.predict(X_test_tfidf)
evaluate_model(y_prio_test, prio_pred, prio_model.classes_, "Priority Classifier")